---
<img style="float: right; margin: 15px 15px 15px 15px;" src="https://images.seeklogo.com/logo-png/51/2/reddit-logo-png_seeklogo-511297.png" width="350px" height="120px" />

# <font color=#bbc28d>**Emotion Detection**</font>
#### <font color=#2E9AFE>`Dataset Preparation-Training Pipeline`</font>

---

This project consists of creating an empathetic chatbot using Transformer models. The chatbot is capable of detecting the emotion behind a user's message and generating a supportive response based on that emotion.

The project uses two different models:

- A **DistilBERT** model for emotion classification.
- A **FLAN-T5** model for response generation.

First, the emotion classification model analyzes the user's text and predicts the emotion expressed in the message. Then, the detected emotion and the original text are sent to the generative model, which creates a natural and empathetic response.


## <font color= #66b0b0> &ensp; • **Part 1: Model for Emotion Detection** </font>

In [ ]:
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, AutoModelForCausalLM, AutoModelForSeq2SeqLM
import torch
import numpy as np

The dataset used for this project is **Empathetic Dialogues**, obtained from Hugging Face.

Each record in the dataset includes:

- A situation described by the user
- The emotion associated with the situation
- A conversation between the user and the assistant

This dataset serves as the foundation for the project, since it provides the emotional context required for training the emotion detection model and generating empathetic responses.

In [ ]:
# Load dataset
ds = load_dataset("Estwld/empathetic_dialogues_llm")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/5.22M [00:00<?, ?B/s]

data/valid-00000-of-00001.parquet:   0%|          | 0.00/806k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/798k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/19533 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/2770 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2547 [00:00<?, ? examples/s]

Before training the emotion classification model, the emotion labels contained in the dataset must be converted into numerical values that can be processed by the neural network.

In [ ]:
# Extract unique labels (emotions)
unique_emotions = sorted(list(set(ds['train']['emotion'])))
emotion2id = {emotion: i for i, emotion in enumerate(unique_emotions)}
id2emotion = {i: emotion for emotion, i in emotion2id.items()}

def preprocess_function(examples):
    # Tokenize the situation
    result = tokenizer(examples["situation"], truncation=True, padding="max_length", max_length=64)
    # Convert the emotion name into an ID
    result["label"] = [emotion2id[emotion] for emotion in examples["emotion"]]
    return result


First, all unique emotions present in the training dataset are extracted and organized alphabetically. Then, two dictionaries are created:


- emotion2id: maps each emotion to a numerical identifier.

- id2emotion: converts numerical predictions back into emotion names.


After defining the labels, a preprocessing function is created. This function tokenizes the text contained in the situation field using the tokenizer associated with the Transformer model. Tokenization converts the text into numerical representations understandable by the model.
Additionally:


- truncation=True limits very long texts.

- padding="max_length" ensures that all inputs have the same length.

- max_length=64 defines the maximum number of tokens.

Finally, the corresponding numerical emotion label is added to each example, preparing the dataset for model training.


--------------------------------------------

We use a tokenizer which is responsible for transforming text into tokens and numerical representations that the neural network can process.

After loading the tokenizer, the preprocessing function defined previously is applied to the entire dataset using the map() method.

The result is a tokenized dataset ready to be used during model training.


In [ ]:
# Tokenize
model_name = "distilbert-base-uncased" # modelo utilizado
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenized_ds = ds.map(preprocess_function, batched=True)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/19533 [00:00<?, ? examples/s]

Map:   0%|          | 0/2770 [00:00<?, ? examples/s]

Map:   0%|          | 0/2547 [00:00<?, ? examples/s]

After preprocessing the dataset, the Transformer model used for emotion classification is initialized and configured.

In this project, the model classifies user inputs into different emotional categories.

The pre-trained model selected is based on distilbert-base-uncased, allowing the project to take advantage of transfer learning and pre-existing language understanding capabilities.
Several parameters are configured:


- num_labels: defines the total number of emotion categories in the dataset.

- id2label: converts numerical predictions into emotion names.

- label2id: converts emotion names into numerical identifiers.


This configuration enables the model to correctly associate each prediction with its corresponding emotion during training and inference.


In [ ]:
# Model configuration
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(unique_emotions),
    id2label=id2emotion,
    label2id=emotion2id)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


These parameters used control different aspects of the training process:

- output_dir specifies the directory where the trained model and checkpoints will be saved.

- eval_strategy="epoch" evaluates the model after each training epoch.

- save_strategy="epoch" saves a checkpoint after every epoch.

- learning_rate=2e-5 defines the step size used by the optimizer during learning.

- per_device_train_batch_size=16 determines the number of samples processed simultaneously during training.

- num_train_epochs=3 we use ony 3 epochs due to the time it takes to run each one.

- weight_decay=0.01 helps reduce overfitting by applying regularization.

- load_best_model_at_end=True automatically restores the best-performing model after training is completed.

These settings were selected to achieve stable training performance while maintaining efficient computational usage in Google Colab.


In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./empathy_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,)

Once the model and training parameters are configured, the `Trainer` class is initialized. This simplifies the training process by managing optimization, evaluation, and data handling automatically.

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["valid"])

In [ ]:
# Training
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.021886,1.718178
2,1.450721,1.513756
3,1.160742,1.497337


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


After training the model, a prediction function is created to classify the emotion expressed in new user inputs.

In [ ]:
# To test
def predecir_emocion(texto):
    inputs = tokenizer(texto, return_tensors="pt", truncation=True, padding=True).to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits
    predicted_class_id = logits.argmax().item()
    return model.config.id2label[predicted_class_id]

We save the model to be able to use it later, without having to train it again.

In [ ]:
# Save trained model
model.save_pretrained("/content/emotion_detection_model")

# Save tokenizer
tokenizer.save_pretrained("/content/emotion_detection_model")

print("Model saved :)")

In [ ]:
import os
os.listdir("/content/emotion_detection_model")

['tokenizer.json', 'model.safetensors', 'config.json', 'tokenizer_config.json']

In [ ]:
# Load model
model_path = "/content/emotion_detection_model"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

print("Modelo cargado correctamente :)")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Modelo cargado correctamente :)


A simple interactive demo was created so that users can write their own phrases and test the emotion detection model

In [ ]:
def probal_modelo_interactivo():
    print("\n" + "="*30)
    print("Emotion classification")
    print("Write your phrase:")
    print("Enter 'exit' or 'quit' to finish the program.")
    print("="*30 + "\n")

    model.eval()

    while True:
        frase = input("Your phrase -> ")

        if frase.lower() in ["salir", "exit", "quit"]:
            print("Bye :)")
            break

        if not frase.strip():
            continue

        # Make prediction
        emocion = predecir_emocion(frase)

        print(f"Emotion detected: {emocion.upper()}")
        print("-" * 20)

probal_modelo_interactivo()


Emotion classification
Write your phrase:
Enter 'exit' or 'quit' to finish the program.

Your phrase -> I have to go out into the woods at night
Emotion detected: AFRAID
--------------------
Your phrase -> exit
Bye :)


--------------------------------
## <font color= #66b0b0> &ensp; • **Part 2:Response Generation** </font>

For the second part of this implementation, we define and load a pre-trained generative model for the chatbot.

We use the model Flan-T5 Base from Google, which is a language model designed for text generation tasks.
Then, we load the tokenizer associated with the model, responsible for converting text into token sequences (numerical representations) that the model can process, and for decoding tokens back into text.

Finally, we load the sequence-to-sequence model (Seq2SeqLM), which generates text outputs based on given inputs.

In [ ]:
# Pre-trained generative model
generator_model_name = "google/flan-t5-base"

generator_tokenizer = AutoTokenizer.from_pretrained(generator_model_name)

generator_model = AutoModelForSeq2SeqLM.from_pretrained(generator_model_name)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


This function generates the chatbot’s reply based on the user’s input.

It first detects the emotion of the message using predecir_emocion.

Then it builds a prompt that instructs the model to act empathetically, including both the emotion and the user’s text.

The prompt is tokenized into numerical form so the model can process it.

Next, the model produces a response with parameters that encourage variety and naturalness.

Finally, the output is decoded back into text, and the function returns both the detected emotion and the generated response.

In [ ]:
# Response generation
def generar_respuesta(user_text):

    # Detect emotion using the first model
    emotion = predecir_emocion(user_text)

    # Create prompt for the second model
    prompt = f"""
You are an empathetic chatbot.

The user emotion is: {emotion}

User message:
{user_text}

Generate a short empathetic response.
"""

    # Tokenize the input
    inputs = generator_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=128)

    # Generate response
    outputs = generator_model.generate(
        **inputs,
        max_length=60,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        top_k=50)

    # Decode generated text
    response = generator_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True)

    return emotion, response

-------------------------------------------------------

## <font color= #66b0b0> &ensp; • **Chat Bot** </font>

Finally, we define the main chatbot loop.

It starts by printing a header with the chatbot’s title and instructions for the user.

Inside a continuous loop, it waits for the user’s input. If the user types "exit" or "quit", the chatbot says goodbye and ends the program. If the input is empty, it simply skips to the next iteration.

For valid input, it calls the function generar_respuesta, which returns both the detected emotion and the generated response.

Finally, it prints the detected emotion and the chatbot’s reply, creating an interactive conversation where the bot responds empathetically to the user.

In [ ]:
# Chatbot final
def empathetic_chatbot():

    print("="*40)
    print("Empathetic Chatbot")
    print("Write 'exit' to finish")
    print("="*40)

    while True:

        user_input = input("\nYou -> ")

        if user_input.lower() in ["exit", "quit"]:
            print("\nBye :)")
            break

        if not user_input.strip():
            continue

        # Generar respuesta
        emotion, response = generar_respuesta(user_input)

        print(f"\nDetected emotion: {emotion}")
        print(f"Bot -> {response}")

In [ ]:
empathetic_chatbot()